Reading CSV file from Catalog

In [0]:
%python
df = spark.read.csv("/Volumes/workspace/default/day1file/C1.csv", 
                    header=True, 
                    inferSchema=True)

In [0]:
%python
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,blank,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,Blank
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,blank,450,C


In [0]:
from pyspark.sql.functions import *
df = df.replace(["blank","Blank"],None)
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,india,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,New York,03-15-2023,400,B
107,Trent,india,10-04-2023,350,B
108,Bob,India,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


In [0]:
from pyspark.sql.functions import upper

df = df.withColumn("Country", upper(col("Country")))
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,INDIA,01-05-2023,150,B
103,Charlie,UK,20-02-2023,null,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,null
106,Mallory,NEW YORK,03-15-2023,400,B
107,Trent,INDIA,10-04-2023,350,B
108,Bob,INDIA,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


In [0]:
df = df.fillna({"Category": "unknown", "Sales": 0})
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,INDIA,01-05-2023,150,B
103,Charlie,UK,20-02-2023,0,C
104,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,unknown
106,Mallory,NEW YORK,03-15-2023,400,B
107,Trent,INDIA,10-04-2023,350,B
108,Bob,INDIA,05-01-2023,150,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,null,450,C


In [0]:
from pyspark.sql.functions import to_date, try_to_date, coalesce

# Try multiple date formats - first format that works is used
df = df.withColumn("JoinDate", 
                   coalesce(
                       try_to_date(col("JoinDate"), "dd-MM-yyyy"),
                       try_to_date(col("JoinDate"), "MM-dd-yyyy")
                   ))

In [0]:
from pyspark.sql.functions import date_format

# Format date as DD-MM-YY
df_display = df.withColumn("JoinDate", date_format(col("JoinDate"), "dd-MM-yy"))
df_display.show()

+----------+-------+--------+--------+-----+--------+
|CustomerID|   Name| Country|JoinDate|Sales|Category|
+----------+-------+--------+--------+-----+--------+
|       101|  Alice|     USA|15-01-23|  250|       A|
|       102|    Bob|   INDIA|01-05-23|  150|       B|
|       103|Charlie|      UK|20-02-23|    0|       C|
|       104|  Alice|     USA|15-01-23|  250|       A|
|       105|    Eve|      UK|01-03-23|  300| unknown|
|       106|Mallory|NEW YORK|15-03-23|  400|       B|
|       107|  Trent|   INDIA|10-04-23|  350|       B|
|       108|    Bob|   INDIA|05-01-23|  150|       B|
|       109|  Oscar|     USA|28-02-23|  500|       A|
|       110|  Peggy|      UK|    NULL|  450|       C|
+----------+-------+--------+--------+-----+--------+



In [0]:
df = df.dropDuplicates()
display(df)

CustomerID,Name,Country,JoinDate,Sales,Category
102,Bob,India,2023-05-01,150,B
103,Charlie,Uk,2023-02-20,0,C
105,Eve,Uk,2023-03-01,300,unknown
101,Alice,Usa,2023-01-15,250,A
109,Oscar,Usa,2023-02-28,500,A
104,Alice,Usa,2023-01-15,250,A
107,Trent,India,2023-04-10,350,B
110,Peggy,Uk,null,450,C
108,Bob,India,2023-01-05,150,B
106,Mallory,New York,2023-03-15,400,B
